# Notebook 20 – Deep Learning Mini Challenge

**Problem chosen:** Predict whether a breast tumour is **malignant or benign** from cell-nucleus measurements.

**Ground rules I set for myself**
- The **test set is touched once**, at the very end.
- Models are compared with **cross-validation**, not one lucky split.
- Network = feed-forward neural network (MLP) built with scikit-learn's `MLPClassifier`, so the notebook runs anywhere with no extra install. The same architecture maps 1:1 to Keras / PyTorch.

# Step 1 – Select a Dataset

## Concept
The Breast Cancer Wisconsin (Diagnostic) dataset: 569 biopsies, 30 numeric measurements of cell nuclei (radius, texture, concavity, etc.), each labelled malignant or benign.

## Why it's needed
The dataset decides what problem is learnable. I chose a high-stakes problem where the two error types are not equal, small enough to train in seconds, and bundled with scikit-learn (works offline).

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer

np.random.seed(42)

data = load_breast_cancer(as_frame=True)
df = data.frame.copy()

# sklearn labels: 0 = malignant, 1 = benign.
# Flip so that 1 = malignant, the class we must not miss.
df["target"] = 1 - df["target"]

print("Source : scikit-learn built-in Breast Cancer Wisconsin (Diagnostic)")
print("Shape  :", df.shape)
print(df.iloc[:3, [0, 1, 2, -1]].to_string())

Source : scikit-learn built-in Breast Cancer Wisconsin (Diagnostic)
Shape  : (569, 31)
   mean radius  mean texture  mean perimeter  target
0        17.99         10.38           122.8       1
1        20.57         17.77           132.9       1
2        19.69         21.25           130.0       1


## What this example tells us
569 rows and 31 columns (30 features + target). sklearn labels malignant as 0, so I flipped it to make **1 = malignant**, the class we must not miss.

## AI/ML Example
Always check which label is 0 and which is 1 before computing recall or precision, otherwise metrics silently measure the wrong class.

## Business Example
A hospital screening-support tool built from past biopsies whose outcomes are already confirmed.

## AI/ML Engineer Use Case
Read the data documentation and verify label direction first. A flipped label is one of the cheapest bugs to make and the hardest to notice.

# Step 2 – Define the Problem

## Concept
Turn the real-world question into an ML task: the task type, the cost of each kind of mistake, and the metric that reflects that cost.

## Why it's needed
The metric drives every later decision. If accuracy is chosen blindly, a model that misses cancers can still look good.

In [2]:
problem = {
    "Task":           "Binary classification",
    "Question":       "Is a tumour malignant (1) or benign (0)?",
    "Costly error":   "False negative (malignant predicted as benign)",
    "Primary metric": "Recall on the malignant class",
    "Support metrics": "Precision, F1, ROC-AUC",
}
for k, v in problem.items():
    print(f"{k:16}: {v}")

Task            : Binary classification
Question        : Is a tumour malignant (1) or benign (0)?
Costly error    : False negative (malignant predicted as benign)
Primary metric  : Recall on the malignant class
Support metrics : Precision, F1, ROC-AUC


## What this example tells us
This is **binary classification**. A false negative (missed cancer) is far worse than a false alarm, so **recall on malignant** is the primary metric, supported by precision, F1 and ROC-AUC.

## AI/ML Example
Cost-sensitive classification: fraud detection, disease screening and fault detection all rank error types unequally.

## Business Example
A missed cancer delays treatment; a false alarm only costs an extra test. The business cost is clearly asymmetric.

## AI/ML Engineer Use Case
Write down the target metric and error costs before training, and agree on them with stakeholders. Do not pick the metric after seeing results.

# Step 3 – Identify Input and Target

## Concept
Input (X) = the 30 numeric measurements. Target (y) = 1 if malignant, 0 if benign.

## Why it's needed
A model can only learn a mapping from inputs to a target. Confirming types, missing values and class counts avoids surprises later.

In [3]:
X = df.drop(columns="target")
y = df["target"]

print("Number of input features:", X.shape[1])
print("First 5 features        :", list(X.columns[:5]))
print("Target counts           :", y.value_counts().to_dict(), "(1 = malignant, 0 = benign)")
print("Rows with missing values:", int(df.isna().any(axis=1).sum()))

Number of input features: 30
First 5 features        : ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness']
Target counts           : {0: 357, 1: 212} (1 = malignant, 0 = benign)
Rows with missing values: 0


## What this example tells us
30 numeric features, **no missing values**, and 357 benign vs 212 malignant cases. No encoding or imputation is needed.

## AI/ML Example
Tabular supervised learning: numeric feature matrix in, one label out.

## Business Example
Each row is one patient's biopsy; the target is the confirmed diagnosis.

## AI/ML Engineer Use Case
Check missing values and dtypes at this stage. It decides how much preprocessing the pipeline needs.

# Step 4 – Perform Basic EDA

## Concept
Exploratory Data Analysis: look at class balance, feature scales and which features relate to the target.

## Why it's needed
EDA tells us what preprocessing and architecture choices the data demands before we spend time training.

In [4]:
print("Class balance (%):")
print((y.value_counts(normalize=True) * 100).round(1).to_string())

print("\nFeature scale (min / max):")
print(X[["mean radius", "mean smoothness", "mean area"]].agg(["min", "max"]).round(3).to_string())

print("\nTop 5 features correlated with malignant:")
print(X.corrwith(y).sort_values(ascending=False).head(5).round(3).to_string())

corr = X.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
print("\nFeature pairs with |correlation| > 0.95:", int((upper.stack() > 0.95).sum()))

Class balance (%):
target
0    62.7
1    37.3

Feature scale (min / max):
     mean radius  mean smoothness  mean area
min        6.981            0.053      143.5
max       28.110            0.163     2501.0

Top 5 features correlated with malignant:
worst concave points    0.794
worst perimeter         0.783
mean concave points     0.777
worst radius            0.776
mean perimeter          0.743

Feature pairs with |correlation| > 0.95: 15


## What this example tells us
Malignant is **37.3%** of the data, so always predicting benign would already score 62.7% accuracy and prove nothing. Features sit on very different scales (area up to 2501, smoothness 0.05 to 0.16), so **scaling is mandatory**. Size and shape features such as `worst concave points` and `worst perimeter` correlate about 0.78 to 0.79 with malignancy, and 15 feature pairs are over 0.95 correlated (redundant information).

## AI/ML Example
Neural networks are sensitive to feature scale. EDA that reveals unequal scales justifies standardisation.

## Business Example
The majority-class baseline of 62.7% is the bar any model must beat to be worth deploying.

## AI/ML Engineer Use Case
Record the baseline accuracy and the strongest features. They are your sanity check for the final model.

# Step 5 – Prepare the Data

## Concept
Split into train / validation / test (60 / 20 / 20, stratified) and standardise features using statistics from the training set only.

## Why it's needed
Validation is for tuning, test is for the final honest score. Fitting the scaler on all data would leak test information into training.

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 60 / 20 / 20 split, stratified so every split keeps the same malignant rate
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=42)

# Fit the scaler on TRAIN only to avoid data leakage
scaler = StandardScaler().fit(X_train)
Xtr, Xva, Xte = scaler.transform(X_train), scaler.transform(X_val), scaler.transform(X_test)

for name, part in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{name:5}: {len(part):3} rows, malignant rate = {part.mean():.3f}")
print("\nTrain mean (should be ~0):", Xtr.mean().round(3))
print("Train std  (should be ~1):", Xtr.std().round(3))

train: 341 rows, malignant rate = 0.372
val  : 114 rows, malignant rate = 0.377
test : 114 rows, malignant rate = 0.368

Train mean (should be ~0): 0.0
Train std  (should be ~1): 1.0


## What this example tells us
All three splits keep about 37% malignant. Train features now have mean about 0 and std 1. The scaler was fit on **train only**.

## AI/ML Example
Leakage-free preprocessing: any fitted transform (scaler, encoder, imputer) must learn from training data only.

## Business Example
A model that leaks looks great in testing and fails on real patients, which is a costly and embarrassing failure.

## AI/ML Engineer Use Case
Put preprocessing inside a Pipeline when possible. It guarantees correct fitting during cross-validation (used in Step 11).

# Step 6 – Select a Neural Network Architecture and Explain Why

## Concept
A feed-forward network (MLP): Input(30) -> Dense(16, ReLU) -> Output(1, Sigmoid). The sigmoid output is the probability of malignant.

## Why it's needed
**Why this architecture:** the data is tabular with 30 independent numeric features, with no image, sequence or spatial structure, so CNNs and RNNs add nothing. With only 341 training rows, a **small** network (513 parameters) limits overfitting. ReLU gives fast, stable learning; sigmoid + log-loss suits a binary target.

In [6]:
from sklearn.neural_network import MLPClassifier

# Config A: baseline - one hidden layer
model_a = MLPClassifier(hidden_layer_sizes=(16,), activation="relu",
                        solver="adam", max_iter=500, random_state=42)

n_in, n_hidden, n_out = Xtr.shape[1], 16, 1
params = (n_in * n_hidden + n_hidden) + (n_hidden * n_out + n_out)
print(f"Input({n_in}) -> Dense({n_hidden}, ReLU) -> Output({n_out}, Sigmoid)")
print("Trainable parameters:", params)
print("Training rows        :", len(Xtr), "| params per row:", round(params / len(Xtr), 2))

Input(30) -> Dense(16, ReLU) -> Output(1, Sigmoid)
Trainable parameters: 513
Training rows        : 341 | params per row: 1.5


## What this example tells us
The baseline has about 1.5 parameters per training row, which is small enough to be safe but big enough to learn non-linear patterns. This is **Config A**.

## AI/ML Example
Match architecture to data type: MLP for tabular, CNN for images, RNN/Transformer for sequences. Size the model to the amount of data.

## Business Example
A small model is cheaper to run, easier to audit and less likely to memorise a few hundred patients.

## AI/ML Engineer Use Case
Always start with the simplest architecture that fits the data type, then justify every extra layer with evidence.

# Step 7 – Train the Model

## Concept
Training repeatedly adjusts weights with the Adam optimiser to reduce log-loss on the training data.

## Why it's needed
The loss curve is the first proof that the model is learning at all.

In [7]:
model_a.fit(Xtr, y_train)

print("Epochs run         :", model_a.n_iter_)
print("Loss at epoch 1    :", round(model_a.loss_curve_[0], 4))
print("Loss at epoch 10   :", round(model_a.loss_curve_[9], 4))
print("Loss at epoch 50   :", round(model_a.loss_curve_[49], 4))
print("Final training loss:", round(model_a.loss_curve_[-1], 4))

Epochs run         : 384
Loss at epoch 1    : 1.4648
Loss at epoch 10   : 1.0758
Loss at epoch 50   : 0.2893
Final training loss: 0.0491


## What this example tells us
Loss dropped from **1.465 to 0.289 by epoch 50** and settled at **0.049** after 384 epochs. A fast early drop followed by slow refinement is a healthy training curve.

## AI/ML Example
Monitoring the loss curve detects problems early: flat curve = not learning, exploding curve = learning rate issue.

## Business Example
Training cost here is seconds, so many experiments can be run cheaply before choosing a final model.

## AI/ML Engineer Use Case
Check the loss at several epochs, not just the last one, and confirm training stopped by convergence rather than hitting the iteration cap.

# Step 8 – Evaluate the Model

## Concept
Compute accuracy, precision, recall, F1, ROC-AUC and the confusion matrix on both train and validation data.

## Why it's needed
Comparing train vs validation shows whether the model generalises or only memorises.

In [8]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

def report(model, X_, y_, label):
    pred = model.predict(X_)
    prob = model.predict_proba(X_)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_, pred).ravel()
    print(f"[{label}] acc={accuracy_score(y_, pred):.3f} "
          f"precision={precision_score(y_, pred):.3f} "
          f"recall={recall_score(y_, pred):.3f} "
          f"f1={f1_score(y_, pred):.3f} "
          f"auc={roc_auc_score(y_, prob):.3f}")
    print(f"          TN={tn} FP={fp} FN={fn} TP={tp}")

report(model_a, Xtr, y_train, "A train")
report(model_a, Xva, y_val,   "A val  ")

[A train] acc=0.988 precision=0.992 recall=0.976 f1=0.984 auc=0.998
          TN=213 FP=1 FN=3 TP=124
[A val  ] acc=0.974 precision=0.955 recall=0.977 f1=0.966 auc=0.996
          TN=69 FP=2 FN=1 TP=42


## What this example tells us
Validation: accuracy **0.974**, recall **0.977**, precision 0.955, AUC 0.996. Train recall (0.976) is close to validation recall, so there is **no serious overfitting**. Validation has 1 missed cancer (FN) and 2 false alarms (FP).

## AI/ML Example
A small train/validation gap means the model has learned patterns that generalise rather than noise.

## Business Example
On 114 validation patients, 1 missed cancer and 2 false alarms is a good start, but 114 rows is a small sample.

## AI/ML Engineer Use Case
Always report the confusion matrix, not only a single score. It shows which error type you are making.

# Step 9 – Perform Error Analysis

## Concept
Inspect the exact rows the model got wrong and look for what they have in common.

## Why it's needed
Metrics say how many errors there are; error analysis says why. It is the source of good improvement ideas.

In [9]:
pred_val = model_a.predict(Xva)
prob_val = model_a.predict_proba(Xva)[:, 1]

errors = X_val.copy()
errors["true"] = y_val.values
errors["pred"] = pred_val
errors["p_malignant"] = prob_val.round(3)
errors = errors[errors["true"] != errors["pred"]]

print("Total errors      :", len(errors), "of", len(X_val))
print("False negatives   :", int(((errors.true == 1) & (errors.pred == 0)).sum()), "(missed cancers)")
print("False positives   :", int(((errors.true == 0) & (errors.pred == 1)).sum()), "(false alarms)")
print("\nMisclassified rows:")
print(errors[["mean radius", "mean concavity", "true", "pred", "p_malignant"]].to_string())

correct = X_val[pred_val == y_val.values]
print("\nMean 'mean radius'    - errors:", round(errors["mean radius"].mean(), 2),
      "| correct:", round(correct["mean radius"].mean(), 2))
print("Mean 'mean concavity' - errors:", round(errors["mean concavity"].mean(), 3),
      "| correct:", round(correct["mean concavity"].mean(), 3))

Total errors      : 3 of 114
False negatives   : 1 (missed cancers)
False positives   : 2 (false alarms)

Misclassified rows:
     mean radius  mean concavity  true  pred  p_malignant
40         13.44         0.03110     1     0        0.076
363        16.50         0.05862     0     1        0.643
238        14.22         0.11030     0     1        0.680

Mean 'mean radius'    - errors: 14.72 | correct: 14.16
Mean 'mean concavity' - errors: 0.067 | correct: 0.096


## What this example tells us
3 errors out of 114. The missed cancer (FN) had a predicted malignant probability of only **0.076**: the model was *confidently wrong*, and the tumour has small radius (13.4) and low concavity (0.031), so it looks benign. The 2 false alarms had probabilities of 0.64 and 0.68, close to the boundary. With only 3 errors, these patterns are hints, not proof.

## AI/ML Example
Confidently-wrong predictions point to model overconfidence or missing information; borderline errors point to threshold or regularisation effects.

## Business Example
Missed cancers that look benign are the hardest cases clinically too. This is where a human second review adds the most value.

## AI/ML Engineer Use Case
Always look at probabilities of the errors, not just the labels. Overconfidence is a different problem from being borderline.

# Step 10 – Improve the Model

## Concept
Change the model in ways motivated by the errors: add L2 regularisation (`alpha`), use early stopping, and try different depth and width.

## Why it's needed
Improvements should be hypotheses from error analysis, not random tuning.

In [10]:
# Improvement ideas driven by the error analysis:
#  B: deeper network + L2 regularisation + early stopping (reduce over-confidence / overfitting)
#  C: wider network + stronger L2 (more capacity, controlled by regularisation)
model_b = MLPClassifier(hidden_layer_sizes=(64, 32), activation="relu", alpha=1e-2,
                        early_stopping=True, validation_fraction=0.15, n_iter_no_change=20,
                        max_iter=500, random_state=42)
model_c = MLPClassifier(hidden_layer_sizes=(128,), activation="relu", alpha=1e-1,
                        max_iter=500, random_state=42)

model_b.fit(Xtr, y_train)
model_c.fit(Xtr, y_train)

print("Epochs run -> A:", model_a.n_iter_, "| B:", model_b.n_iter_, "| C:", model_c.n_iter_)
report(model_a, Xva, y_val, "A val")
report(model_b, Xva, y_val, "B val")
report(model_c, Xva, y_val, "C val")

Epochs run -> A: 384 | B: 62 | C: 272
[A val] acc=0.974 precision=0.955 recall=0.977 f1=0.966 auc=0.996
          TN=69 FP=2 FN=1 TP=42
[B val] acc=0.991 precision=1.000 recall=0.977 f1=0.988 auc=0.997
          TN=71 FP=0 FN=1 TP=42
[C val] acc=0.974 precision=0.955 recall=0.977 f1=0.966 auc=0.996
          TN=69 FP=2 FN=1 TP=42


## What this example tells us
**Config B** (64-32, alpha 0.01, early stopping) reached validation accuracy 0.991 and precision 1.000. **Config C** (128 units, alpha 0.1) matched A exactly. A and C each had 1 missed cancer. B looks best, but the validation set is only 114 rows, so one patient changes the picture. We must not trust this yet.

## AI/ML Example
Regularisation and early stopping are the standard tools to reduce overconfidence and overfitting in small-data networks.

## Business Example
A model change that looks better on one small split may not be better in production. We need a stronger comparison before spending money on it.

## AI/ML Engineer Use Case
Never declare a winner from a single small validation split. Move to cross-validation (next step).

# Step 11 – Compare At Least Two Configurations

## Concept
Compare configurations A, B and C using 5-fold stratified cross-validation on train + validation, with the scaler inside the pipeline.

## Why it's needed
Cross-validation gives a mean and a spread across 5 different splits, which is much more reliable than a single 114-row split.

In [11]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate

# One validation split is noisy (only 114 rows), so compare with 5-fold CV on train + val.
# The scaler sits inside the pipeline, so it is refit per fold (no leakage).
X_dev = pd.concat([X_train, X_val])
y_dev = pd.concat([y_train, y_val])

configs = {
    "A (16)":         dict(hidden_layer_sizes=(16,)),
    "B (64-32, es)":  dict(hidden_layer_sizes=(64, 32), alpha=1e-2, early_stopping=True,
                           validation_fraction=0.15, n_iter_no_change=20),
    "C (128, a=0.1)": dict(hidden_layer_sizes=(128,), alpha=1e-1),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rows = []
for name, cfg in configs.items():
    pipe = make_pipeline(StandardScaler(), MLPClassifier(max_iter=500, random_state=42, **cfg))
    res = cross_validate(pipe, X_dev, y_dev, cv=cv,
                         scoring=["recall", "precision", "f1", "roc_auc"])
    rows.append({"config": name,
                 "recall":    f"{res['test_recall'].mean():.3f} +/- {res['test_recall'].std():.3f}",
                 "precision": f"{res['test_precision'].mean():.3f} +/- {res['test_precision'].std():.3f}",
                 "f1":        f"{res['test_f1'].mean():.3f} +/- {res['test_f1'].std():.3f}",
                 "auc":       f"{res['test_roc_auc'].mean():.3f} +/- {res['test_roc_auc'].std():.3f}"})
print(pd.DataFrame(rows).to_string(index=False))

        config          recall       precision              f1             auc
        A (16) 0.953 +/- 0.014 0.954 +/- 0.039 0.953 +/- 0.023 0.994 +/- 0.006
 B (64-32, es) 0.853 +/- 0.072 0.963 +/- 0.043 0.902 +/- 0.044 0.982 +/- 0.010
C (128, a=0.1) 0.953 +/- 0.014 0.977 +/- 0.022 0.964 +/- 0.007 0.993 +/- 0.007


## What this example tells us
B, which looked best earlier, is actually the **worst**: recall 0.853 +/- 0.072 vs 0.953 for A and C. Early stopping holds out another 15% of already small data, which hurts. **C** has the best precision (0.977), best F1 (0.964) and the lowest variance (+/- 0.007).

## AI/ML Example
A configuration that wins on one split but loses in cross-validation is a classic sign of validation noise. This is why CV is the default for small datasets.

## Business Example
Choosing B would have meant shipping a model that misses about 1 in 7 cancers instead of about 1 in 21.

## AI/ML Engineer Use Case
Compare on mean AND standard deviation. A stable model is usually safer than a slightly higher but erratic one.

# Step 12 – Select the Final Model

## Concept
Apply a selection rule fixed in advance (highest CV recall, then CV F1, then lowest variance), refit on train + validation, and score the test set once.

## Why it's needed
The test set is the only unbiased estimate of real-world performance. Using it for selection would destroy that.

In [12]:
# Selection rule (decided BEFORE touching the test set):
# highest CV recall -> tie-break on CV F1 -> tie-break on lowest variance.
# A and C tie on recall (0.953); C has the better F1 (0.964 vs 0.953) and lower variance.
# B looked best on the single validation split but was worst in CV, so it is rejected.
final_model = make_pipeline(StandardScaler(),
                            MLPClassifier(hidden_layer_sizes=(128,), alpha=1e-1,
                                          max_iter=500, random_state=42))
final_model.fit(X_dev, y_dev)          # train + validation, test untouched until now

report(final_model, X_test, y_test, "FINAL test")

pred_test = final_model.predict(X_test)
prob_test = final_model.predict_proba(X_test)[:, 1]
wrong = X_test[pred_test != y_test.values][["mean radius", "mean concavity"]].copy()
wrong["true"] = y_test[pred_test != y_test.values].values
wrong["p_malignant"] = prob_test[pred_test != y_test.values].round(3)
print("\nMisclassified test rows:")
print(wrong.to_string())

[FINAL test] acc=0.974 precision=1.000 recall=0.929 f1=0.963 auc=0.991
          TN=72 FP=0 FN=3 TP=39

Misclassified test rows:
     mean radius  mean concavity  true  p_malignant
73         13.80         0.07789     1        0.004
190        14.22         0.19810     1        0.015
205        15.12         0.07550     1        0.321


## What this example tells us
**Config C** wins (tie with A on recall, better F1 and variance). Final test result: accuracy **0.974**, precision **1.000**, recall **0.929**, F1 0.963, AUC 0.991, with 0 false alarms and **3 missed cancers**. Test recall is lower than CV recall (0.953), which is plausible given the small sample. The 3 misses had malignant probabilities of 0.004, 0.015 and 0.321, so two were confidently wrong.

## AI/ML Example
Reporting the test number honestly, even when it is lower than hoped, is what makes a result trustworthy.

## Business Example
Perfect precision but 3 missed cancers means the model is cautious about raising alarms, which is the wrong direction for screening. It needs a human in the loop.

## AI/ML Engineer Use Case
Fix the selection rule before looking at the test set, and report the test score exactly once.

# Step 13 – Explain Limitations

## Concept
State what the result does and does not prove: sample size, uncertainty, stability and data scope.

## Why it's needed
Every model result has uncertainty; hiding it is how overconfident systems get deployed.

In [13]:
rng = np.random.default_rng(42)
y_t, p_t = y_test.values, pred_test

# 1) How uncertain is the test score? Bootstrap 95% interval.
recalls, accs = [], []
for _ in range(1000):
    idx = rng.integers(0, len(y_t), len(y_t))
    if y_t[idx].sum() == 0:
        continue
    recalls.append(recall_score(y_t[idx], p_t[idx]))
    accs.append(accuracy_score(y_t[idx], p_t[idx]))
print("Malignant cases in test set:", int(y_t.sum()))
print("Recall   95% interval:", np.percentile(recalls, [2.5, 97.5]).round(3))
print("Accuracy 95% interval:", np.percentile(accs, [2.5, 97.5]).round(3))

# 2) Is the result stable across random weight initialisations?
fn_list = []
for seed in range(5):
    m = make_pipeline(StandardScaler(),
                      MLPClassifier(hidden_layer_sizes=(128,), alpha=1e-1,
                                    max_iter=500, random_state=seed)).fit(X_dev, y_dev)
    fn_list.append(int(((m.predict(X_test) == 0) & (y_test.values == 1)).sum()))
print("\nFalse negatives across 5 seeds:", fn_list)

Malignant cases in test set: 42
Recall   95% interval: [0.842 1.   ]
Accuracy 95% interval: [0.939 1.   ]

False negatives across 5 seeds: [3, 4, 4, 3, 3]


## What this example tells us
The test set has only **42 malignant cases**. The bootstrap 95% interval for recall is **0.842 to 1.000**, so true performance could be well below 0.93. Across 5 random seeds, missed cancers ranged from 3 to 4, so results are fairly stable but not identical. Other limits: one dataset from one source with only 569 rows, no external validation, features are pre-computed measurements (not raw images), and no non-neural baseline (e.g. logistic regression) was compared, so we cannot claim the neural network is *needed* here.

## AI/ML Example
On small tabular data, simpler models often match neural networks. A fair baseline comparison is part of honest evaluation.

## Business Example
This model is a research prototype, not a diagnostic device. Clinical use would need external validation and regulatory approval.

## AI/ML Engineer Use Case
Quantify uncertainty with bootstrap or CV intervals and always state it next to the headline metric.

# Step 14 – Suggest Future Improvements

## Concept
Concrete next steps ranked by expected impact, with at least one tested quickly to check it is worthwhile.

## Why it's needed
Good improvement plans are specific and evidence-based, not a generic wishlist.

In [14]:
# Preview of one improvement: move the decision threshold to trade false alarms for fewer missed cancers.
# Chosen on the VALIDATION set using model C (trained on train only) - not applied to the final model.
prob_va = model_c.predict_proba(Xva)[:, 1]
rows = []
for t in [0.5, 0.4, 0.3, 0.2, 0.1]:
    pred = (prob_va >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, pred).ravel()
    rows.append({"threshold": t, "recall": round(recall_score(y_val, pred), 3),
                 "precision": round(precision_score(y_val, pred), 3),
                 "missed_cancers(FN)": fn, "false_alarms(FP)": fp})
print(pd.DataFrame(rows).to_string(index=False))

 threshold  recall  precision  missed_cancers(FN)  false_alarms(FP)
       0.5   0.977      0.955                   1                 2
       0.4   0.977      0.913                   1                 4
       0.3   0.977      0.913                   1                 4
       0.2   0.977      0.875                   1                 6
       0.1   0.977      0.824                   1                 9


## What this example tells us
I tested **threshold lowering** on validation: recall stayed at **0.977** (1 miss) while false alarms rose from 2 to 9 as the threshold dropped from 0.5 to 0.1. Lowering the threshold does not rescue *confidently wrong* cases, so the fix must come from better data or models, not threshold alone. **Next steps:** (1) more patients and an external validation set; (2) add a logistic regression / gradient boosting baseline; (3) probability calibration to reduce overconfidence; (4) cost-sensitive loss that penalises missed cancers more; (5) a CNN on the raw biopsy images; (6) feature-importance / SHAP explanations for clinicians.

## AI/ML Example
When a cheap fix (threshold) fails, the problem is usually in the data or the model's information, which directs effort to the right place.

## Business Example
More data and external validation reduce business risk the most; threshold tuning would only increase false alarms and cost.

## AI/ML Engineer Use Case
Test the cheapest idea first, record what it showed, and let the result decide the next investment.